in terminal run:
#pip install pandas

In [7]:
import pandas as pd
from pathlib import Path

# Read data

In [8]:
df = pd.read_csv("MBL Labs Interview Materials/sales_data_raw.csv")

# See the first 10 rows
print(df.head(10))

# See column names and data types
print(df.dtypes)

# See how many rows and columns you have
print(df.shape)

   order_id         order_date customer_id         customer_name region  \
0      1001         03/01/2024       C0042  Hartley Supplies Ltd  North   
1      1002         05/01/2024       C0017         Brennan & Co.  South   
2      1003         05/01/2024       C0091                   NaN   East   
3      1004         08/01/2024       C0042  Hartley Supplies Ltd  North   
4      1005   9th January 2024       C0055        Pinnacle Media   West   
5      1006         10/01/2024       C0033      Morrison Digital  South   
6      1007         15/01/2024       C0078         Ashford Group   East   
7      1008         17/01/2024       C0091                   NaN   East   
8      1009         19/01/2024       C0019      Langley Partners  North   
9      1010  22nd January 2024       C0055        Pinnacle Media   West   

  product_category              product_name  quantity  unit_price  \
0  Office Supplies           Stapler Pro 200         3       12.99   
1       Technology         Wireles

# Visible issues:

- `order_date` shows as `object` (string), not a proper date format
- Some rows in `customer_name` look blank or contain "N/A",
- unify customer name 
- `discount_pct` is an integer
- unify Region

in addition:
- check customer name
- check duplicated rows
- no total revenue, no currency, no flag for promotion

In [9]:
# Issue 1: Mixed date formats
print("--- 1.Dates that look unusual ---")
unusual_dates = df[~df["order_date"].str.match(r"^\d{2}/\d{2}/\d{4}$", na=False)]
print(unusual_dates[["order_id", "order_date"]])

--- 1.Dates that look unusual ---
    order_id          order_date
4       1005    9th January 2024
9       1010   22nd January 2024
13      1014   3rd February 2024
17      1018  11th February 2024
18      1019  13th February 2024
23      1024  27th February 2024
24      1025      1st March 2024


In [10]:
# Issue 2 & 7: Missing or inconsistent customer names for C0091
print("--- 2.All rows for customer C0091 ---")
print(df[df["customer_id"] == "C0091"][["order_id", "customer_id", "customer_name"]])
print(df[df["customer_id"] == "C0055"][["order_id", "customer_id", "customer_name"]])

--- 2.All rows for customer C0091 ---
    order_id customer_id    customer_name
2       1003       C0091              NaN
7       1008       C0091              NaN
22      1023       C0091              NaN
31      1032       C0091  Greenfield Corp
41      1042       C0091  Greenfield Corp
49      1050       C0091  Greenfield Corp
57      1058       C0091  Greenfield Corp
65      1066       C0091  Greenfield Corp
73      1074       C0091  Greenfield Corp
82      1083       C0091  Greenfield Corp
90      1091       C0091  Greenfield Corp
98      1099       C0091  Greenfield Corp
    order_id customer_id   customer_name
4       1005       C0055  Pinnacle Media
9       1010       C0055  Pinnacle Media
18      1019       C0055  Pinnacle media
24      1025       C0055  Pinnacle Media
32      1033       C0055  Pinnacle Media
39      1040       C0055  Pinnacle Media
48      1049       C0055  Pinnacle Media
56      1057       C0055  Pinnacle Media
64      1065       C0055  Pinnacle Media
72    

In [11]:
# Issue 3: Inconsistent region casing
print("--- 3.Unique region values ---")
print(df["region"].value_counts())

--- 3.Unique region values ---
region
North    27
West     25
South    24
East     23
NORTH     1
Name: count, dtype: int64


In [12]:
# Issue 4: Inconsistent customer_name casing
print("--- 4.Unique customer names (sorted) ---")
print(sorted(df["customer_name"].dropna().unique()))

--- 4.Unique customer names (sorted) ---
['Ashford Group', 'Brennan & Co.', 'Greenfield Corp', 'Hartley Supplies Ltd', 'Langley Partners', 'Morrison Digital', 'Nexus Retail', 'Pinnacle Media', 'Pinnacle media']


In [13]:
# Issues 5 & 6: Exact duplicate rows
print("--- 5.Checking for duplicate rows ---")
dupes = df[df.duplicated(subset=["customer_id", "order_date", "product_name", "sales_rep", "quantity", "unit_price"], keep=False)]
print(dupes[["order_id", "order_date", "customer_id", "product_name", "sales_rep", "quantity", "unit_price"]])

--- 5.Checking for duplicate rows ---
    order_id  order_date customer_id       product_name       sales_rep  \
14      1015  06/02/2024       C0019  Standing Desk Pro  Sarah Mitchell   
15      1016  06/02/2024       C0019  Standing Desk Pro  Sarah Mitchell   

    quantity  unit_price  
14         1       599.0  
15         1       599.0  


# Data Cleaning

In [ ]:
## Step 1: Remove suffixes from order_date(1st, 2nd, 3rd, 4th, etc.)
df["order_date"] = (
    df["order_date"]
    .astype(str)
    .str.strip()
    .str.replace(r"(\d{1,2})(st|nd|rd|th)", r"\1", regex=True)
)

    order_id  order_date
4       1005  2024-01-09
9       1010  2024-01-22
13      1014  2024-02-03


In [18]:
# Step 2: Set proper datetime, then format as YYYY-MM-DD
df["order_date"] = pd.to_datetime(
    df["order_date"], format="mixed", dayfirst=True, errors="coerce"
).dt.strftime("%Y-%m-%d")

# Verify: check for any rows that failed to parse (should be 0)
print("Unparsed dates:", df["order_date"].isna().sum())

# Spot-check the previously unusual rows
print(df[df["order_id"].isin([1005, 1010, 1014])][["order_id", "order_date"]])

Unparsed dates: 0
    order_id  order_date
4       1005  2024-09-01
9       1010  2024-01-22
13      1014  2024-03-02


In [9]:
# Step 4 — Fix region casing (Issue 3)

df["region"] = df["region"].astype(str).str.strip().str.title()

# Verify: should only show North, South, East, West
print(df["region"].value_counts())

region
North    28
West     25
South    24
East     23
Name: count, dtype: int64


In [10]:
## Step 5 — Fix customer_name casing (Issue 4)

df["customer_name"] = (
    df["customer_name"]
    .fillna("")          # replace actual NaN with empty string first
    .astype(str)
    .str.strip()
    .str.title()
)

# Verify Pinnacle Media rows
print(df[df["customer_id"] == "C0055"][["order_id", "customer_name"]].drop_duplicates())


    order_id   customer_name
4       1005  Pinnacle Media
9       1010  Pinnacle Media
18      1019  Pinnacle Media
24      1025  Pinnacle Media
32      1033  Pinnacle Media
39      1040  Pinnacle Media
48      1049  Pinnacle Media
56      1057  Pinnacle Media
64      1065  Pinnacle Media
72      1073  Pinnacle Media
81      1082  Pinnacle Media
89      1090  Pinnacle Media
97      1098  Pinnacle Media


In [ ]:
## Step 6 — Resolve C0091 customer name (Issues 2 & 7)
# Assumption: "Greenfield Corp" is used as the canonical name because it is the
# actual trading name that appears in rows where customer_name was correctly populated.

canonical = "Greenfield Corp"

df.loc[df["customer_id"] == "C0091", "customer_name"] = canonical

# Verify: all C0091 rows should now show Greenfield Corp
print(df[df["customer_id"] == "C0091"][["order_id", "customer_name"]])


    order_id    customer_name
2       1003  Greenfield Corp
7       1008  Greenfield Corp
22      1023  Greenfield Corp
31      1032  Greenfield Corp
41      1042  Greenfield Corp
49      1050  Greenfield Corp
57      1058  Greenfield Corp
65      1066  Greenfield Corp
73      1074  Greenfield Corp
82      1083  Greenfield Corp
90      1091  Greenfield Corp
98      1099  Greenfield Corp


In [12]:
## Step 7 — Remove duplicate rows (Issues 5 & 6)
rows_before = len(df)

duplicate_ids_to_drop = [1016, 1076]
df = df[~df["order_id"].isin(duplicate_ids_to_drop)].copy()

rows_after = len(df)
print(f"Rows before: {rows_before}")
print(f"Rows removed: {rows_before - rows_after}")
print(f"Rows after: {rows_after}")


Rows before: 100
Rows removed: 2
Rows after: 98


# Calculation

In [20]:
## Step 8 — Calculate revenue
# revenue = quantity × unit_price × (1 − discount_pct / 100)

df["revenue"] = (
    df["quantity"] * df["unit_price"] * (1 - df["discount_pct"] / 100)
).round(2)

print(df.head(10))
# Spot-check a row you can verify manually
# Row 1002: qty=5, unit_price=24.50, discount=10%  →  5 * 24.50 * 0.90 = 110.25
print(df[df["order_id"] == 1002][["order_id", "quantity", "unit_price", "discount_pct", "revenue"]])

   order_id  order_date customer_id         customer_name region  \
0      1001  2024-03-01       C0042  Hartley Supplies Ltd  North   
1      1002  2024-05-01       C0017         Brennan & Co.  South   
2      1003  2024-05-01       C0091       Greenfield Corp   East   
3      1004  2024-08-01       C0042  Hartley Supplies Ltd  North   
4      1005  2024-09-01       C0055        Pinnacle Media   West   
5      1006  2024-10-01       C0033      Morrison Digital  South   
6      1007  2024-01-15       C0078         Ashford Group   East   
7      1008  2024-01-17       C0091       Greenfield Corp   East   
8      1009  2024-01-19       C0019      Langley Partners  North   
9      1010  2024-01-22       C0055        Pinnacle Media   West   

  product_category              product_name  quantity  unit_price  \
0  Office Supplies           Stapler Pro 200         3       12.99   
1       Technology         Wireless Mouse X1         5       24.50   
2        Furniture           Ergonomic Ch

# Final check and save

In [14]:
## Step 9 — Final check and save

# Sort by order_id for clean output

df = df.sort_values("order_id").reset_index(drop=True)

# Quality summary
print("=== Final Data Quality Summary ===")
print(f"Total rows: {len(df)}")
print(f"Null dates: {df['order_date'].isna().sum()}")
print(f"Null customer names: {(df['customer_name'] == '').sum()}")
print(f"Unique regions: {sorted(df['region'].unique())}")
print(f"Revenue range: £{df['revenue'].min()} – £{df['revenue'].max()}")
print(f"Total revenue: £{df['revenue'].sum().round(2)}")

# Save
df.to_csv("sales_data_cleaned.csv", index=False)
print("\nSaved: sales_data_cleaned.csv")

=== Final Data Quality Summary ===
Total rows: 98
Null dates: 0
Null customer names: 0
Unique regions: ['East', 'North', 'South', 'West']
Revenue range: £25.98 – £1797.0
Total revenue: £29900.89

Saved: sales_data_cleaned.csv
